# Urdu ASR Benchmark: Multi-Model Evaluation on Common Voice

## Overview
This notebook benchmarks multiple Automatic Speech Recognition (ASR) models on Urdu audio from the **Mozilla Common Voice 24.0** corpus. The pipeline is split into two stages:

**Stage 1 — Dataset Preparation**
1. Load and filter the validated Common Voice TSV by community vote quality
2. Build a high-frequency Urdu vocabulary from the filtered sentences
3. Run a greedy vocabulary-maximized sampling to select 3,000 representative sentences

**Stage 2 — Multi-Model Inference** *(memory-efficient: one model at a time)*
- `facebook/seamless-m4t-v2-large` — Seamless M4T v2 Large
- `openai/whisper-large-v3` — Whisper Large v3 (with resumable checkpointing)
- `openai/whisper-medium` — Whisper Medium
- `kingabzpro/wav2vec2-large-xls-r-300m-Urdu` — Wav2Vec2 Urdu (CTC)

**Final output:** `all_models_combined.csv` — ground-truth sentences alongside all four model predictions, ready for WER/CER evaluation.

---
**Dataset:** Mozilla Common Voice 24.0 — Urdu (`ur`)  
**Hardware:** GPU strongly recommended (CUDA)

## 1. Imports

Core libraries used throughout the notebook:
- `pandas` / `numpy` — data loading and manipulation
- `regex` — Unicode-aware Urdu text processing
- `collections.Counter` — word frequency counting

In [ ]:
import pandas as pd
import numpy as np
import regex as re
from collections import Counter

## 2. Dataset Paths

Paths to the two TSV files from the Common Voice Urdu corpus:
- `validated.tsv` — sentence metadata and community votes for all validated recordings
- `clip_durations.tsv` — per-clip audio duration (available for duration-based filtering if needed)

In [ ]:
# Path to the main validated recordings metadata file
validated_dataset_path = '/kaggle/input/datasets/elatedspider/coral-urdu-dataset/cv-corpus-24.0-2025-12-05/ur/validated.tsv'

# Path to the clip durations file (available for duration-based filtering if needed)
clip_duration_dataset_path = '/kaggle/input/datasets/elatedspider/coral-urdu-dataset/cv-corpus-24.0-2025-12-05/ur/clip_durations.tsv'

## 3. Load Validated Dataset

Define explicit column dtypes before reading to avoid mixed-type inference issues, then load the TSV into a DataFrame.

In [ ]:
# Explicit dtype mapping for all columns in validated.tsv.
# Nullable Int64 is used for vote columns to correctly handle missing values.
dtypes_validated_tsv = {
    "client_id"       : "string",
    "path"            : "string",
    "sentence_id"     : "string",
    "sentence"        : "string",
    "sentence_domain" : "string",
    "up_votes"        : "Int64",
    "down_votes"      : "Int64",
    "age"             : "string",
    "gender"          : "string",
    "accents"         : "string",
    "variant"         : "string",
    "locale"          : "string",
    "segment"         : "string"
}

In [ ]:
# Load the validated TSV with explicit dtypes to ensure correct column parsing
validated_df = pd.read_csv(validated_dataset_path, sep='\t', dtype=dtypes_validated_tsv)

## 4. Quality Filtering

Retain only recordings with strong community agreement to remove noisy or disputed clips:
- **`up_votes >= 4`** — at least 4 positive validations
- **`down_votes <= 2`** — at most 2 rejections

In [ ]:
# Filter to high-quality recordings based on community vote thresholds
filtered_df = validated_df[
    (validated_df['up_votes'] >= 4) & 
    (validated_df['down_votes'] <= 2)
].copy()

print(f"Original: {len(validated_df):,} → Filtered: {len(filtered_df):,}")

## 5. Urdu Text Tokenizer

A lightweight tokenizer tailored for Urdu script that:
1. Strips common English punctuation
2. Strips Urdu-specific punctuation (`۔`, `،`, `؛`, `؟`)
3. Keeps only characters within the Urdu Unicode block (`U+0600–U+06FF`)
4. Returns a clean list of word tokens

In [ ]:
def tokenize_urdu(text):
    """
    Tokenize an Urdu sentence into a list of clean word tokens.

    Steps:
        1. Remove common English punctuation characters.
        2. Remove Urdu-specific punctuation (U+06D4, U+060C, U+061B, U+061F).
        3. Strip any characters outside the Urdu Unicode block (U+0600–U+06FF).
        4. Split on whitespace and return non-empty tokens.

    Args:
        text (str): Raw Urdu sentence string.

    Returns:
        list[str]: List of Urdu word tokens with punctuation removed.
    """
    # Remove English punctuation marks
    text = re.sub(r'[.,?!"\'\-\(\)\[\]\{\}/\\:;]', ' ', text)

    # Remove Urdu punctuation: ۔ ، ؛ ؟
    text = re.sub(r'[\u06D4\u060C\u061B\u061F]', ' ', text)

    # Retain only characters in the Urdu Unicode block (U+0600–U+06FF)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)

    # Split on whitespace and filter out empty strings
    words = [w.strip() for w in text.split() if w.strip()]
    return words

# Sanity check: tokenize a sample Urdu sentence
print(tokenize_urdu('?موسیقی ,کانوں کو بھلی لگتی ہو۔'))

## 6. Word Frequency Analysis

Tokenize every sentence in the filtered dataset and compute word frequencies. This vocabulary is used to drive the vocabulary-maximized sampling strategy in the next section.

In [ ]:
# Minimum number of occurrences for a word to be considered "high-frequency"
frequency_cutoff = 3

# Collect all tokens from every sentence in the quality-filtered dataset
all_words = []
for sentence in filtered_df['sentence']:
    words = tokenize_urdu(sentence)
    all_words.extend(words)

# Count how often each unique word appears across all sentences
word_counts = Counter(all_words)

print(f"Total words: {len(all_words):,}")
print(f"Unique words: {len(word_counts):,}")

## 7. Build High-Frequency Vocabulary

Filter the full word list to retain only words that meet the frequency cutoff. This set is the target vocabulary we aim to maximally cover in the 3,000-sentence sample.

In [ ]:
# Build the target vocabulary: only words appearing >= frequency_cutoff times
high_freq_vocab = {word for word, count in word_counts.items() if count >= frequency_cutoff}

print(f"High-frequency vocabulary: {len(high_freq_vocab)} words")

## 8. Annotate DataFrame with Vocabulary Features

For each sentence, compute three helper columns used by the greedy sampler:
- `words` — full token list for the sentence
- `high_freq_words` — subset of tokens that appear in the high-frequency vocabulary
- `high_freq_count` — number of unique high-freq vocabulary words in the sentence

Sorting by `high_freq_count` descending ensures the greediest sentences (most vocabulary bang-per-clip) are evaluated first.

In [ ]:
# Reset index after quality filtering so positional (loc) indexing is correct
filtered_df = filtered_df.reset_index(drop=True)

# Tokenize each sentence and store the token list
filtered_df['words'] = filtered_df['sentence'].apply(tokenize_urdu)

# Identify which tokens in each sentence belong to the high-frequency vocabulary
filtered_df['high_freq_words'] = filtered_df['words'].apply(
    lambda words: set(w for w in words if w in high_freq_vocab)
)

# Count unique high-frequency vocabulary words per sentence (greedy sort key)
filtered_df['high_freq_count'] = filtered_df['high_freq_words'].apply(len)

# Sort descending so sentences covering the most vocabulary appear first
filtered_df = filtered_df.sort_values('high_freq_count', ascending=False).reset_index(drop=True)

## 9. Vocabulary-Maximized Greedy Sampling

A two-phase greedy selection strategy to build a 3,000-sentence benchmark sample:

**Phase 1 — Full Coverage Pass:**  
Iterate sentences sorted by `high_freq_count` descending. Add a sentence only if it introduces at least one **new** vocabulary word not yet covered. Stop early once every high-frequency word is seen at least once.

**Phase 2 — Fill to 3,000:**  
From the remaining unselected sentences (again sorted by coverage), take as many as needed to reach the 3,000 target.

In [ ]:
# --- Phase 1: Greedy vocabulary coverage pass ---
# Track which high-frequency words have been seen so far
covered_words = set()
selected_indices = []

for idx in range(len(filtered_df)):
    sentence_words = filtered_df.loc[idx, 'high_freq_words']
    new_words = sentence_words - covered_words  # Words this sentence adds

    if len(new_words) > 0:
        selected_indices.append(idx)
        covered_words.update(new_words)

        # Early exit once every high-frequency word has been covered
        if len(covered_words) == len(high_freq_vocab):
            print(f"Full coverage with {len(selected_indices)} sentences")
            break

# --- Phase 2: Fill remaining slots to reach the 3,000 target ---
# Build pool of unselected sentences sorted by high-freq coverage descending
remaining_pool = [i for i in range(len(filtered_df)) if i not in selected_indices]
remaining_pool_sorted = sorted(
    remaining_pool,
    key=lambda i: filtered_df.loc[i, 'high_freq_count'],
    reverse=True
)

# Append enough sentences to reach the 3,000 total
needed = 3000 - len(selected_indices)
selected_indices.extend(remaining_pool_sorted[:needed])

print(f"Total sampled: {len(selected_indices)}")
print(f"Coverage: {len(covered_words)}/{len(high_freq_vocab)}")

## 10. Extract and Save the Sample

Select only the original Common Voice columns (dropping computed helper columns) and save the final 3,000-sentence sample as a TSV file for use in the inference pipeline.

In [ ]:
# Keep only original Common Voice columns; drop computed helper columns
original_columns = [
    'client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
    'up_votes', 'down_votes', 'age', 'gender', 'accents',
    'variant', 'locale', 'segment'
]

# Extract the sampled rows using the selected indices
sampled_df = filtered_df.loc[selected_indices, original_columns].copy()

# Persist the vocabulary-maximized sample for the inference pipeline
sampled_df.to_csv('sampled_3000_vocab_maximized.tsv', sep='\t', index=False)
print("✓ Sample saved!")

---

## 11. ASR Inference Pipeline — Configuration & Imports

This section initializes the inference environment:
- Imports all model classes from `transformers`, `torch`, `torchaudio`, and `librosa`
- Detects GPU availability and sets the target device
- Configures **TEST MODE** (small sample for pipeline validation) vs **FULL MODE** (all 3,000 sentences)

> **Memory strategy:** Each model is loaded, run to completion, results saved, and immediately deleted before the next model is loaded. This avoids GPU OOM errors when running multiple large models sequentially.

In [ ]:
import pandas as pd
import torch
import torchaudio
import librosa
import numpy as np
from tqdm import tqdm
from transformers import AutoProcessor
from transformers import (
    # Seamless M4T models
    AutoProcessor as SeamlessProcessor,
    SeamlessM4TModel,
    SeamlessM4Tv2Model,

    # Whisper models
    WhisperProcessor,
    WhisperForConditionalGeneration,

    # Wav2Vec2 models
    Wav2Vec2Processor,
    Wav2Vec2ForCTC
)

# -------------------------------------------------------------------
# TEST MODE: set to True to run on a small random sample before the
# full pipeline — useful for validating audio loading and model I/O
# -------------------------------------------------------------------
test_pipeline_mode = False
test_pipeline_on_samples = 3

# Detect GPU availability; fall back to CPU if CUDA is not present
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Base directory containing the Common Voice audio clips
audio_folder = '/kaggle/input/datasets/elatedspider/coral-urdu-dataset/cv-corpus-24.0-2025-12-05/ur/clips/'

# Select the dataset slice based on the configured mode
if test_pipeline_mode:
    # Use a small random sample to validate the pipeline end-to-end
    test_df = sampled_df.sample(n=test_pipeline_on_samples, random_state=42)
    print(f"🧪 TEST MODE: Running on {len(test_df)} random samples")
else:
    # Full production run on all 3,000 vocabulary-maximized sentences
    test_df = sampled_df
    print(f"🚀 FULL MODE: Running on all {len(test_df)} samples")

print(f"\nSample sentences:")
for idx, row in test_df.head(3).iterrows():
    print(f"  - {row['sentence'][:60]}...")

## 12. Model 1 — Seamless M4T v2 Large

**Model:** `facebook/seamless-m4t-v2-large`  
**Task:** Speech-to-text, target language `urd` (Urdu)

Pipeline per clip:
1. Load audio with `torchaudio` (preserves original sample rate)
2. Resample to 16 kHz if needed
3. Convert to mono by averaging channels
4. Run encoder-decoder generation with `tgt_lang="urd"`
5. Decode output token IDs to a string

Results are saved to `seamless_large_results.csv`, then the model is deleted.

In [ ]:
print("\n" + "="*80)
print("MODEL 1: Seamless-Large")
print("="*80)

# Load the Seamless M4T v2 Large processor and model onto the target device
print("Loading Seamless-Large...")
seamless_large_processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
seamless_large_model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large").to(device)
seamless_large_model.eval()
print("✓ Model loaded\n")

# Run inference across the full test set
print("Running inference...")
seamless_large_results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):

    audio_path = f"{audio_folder}{row['path']}"
    ground_truth = row['sentence']

    try:
        # Load audio with torchaudio (preserves original sample rate)
        audio, orig_sr = torchaudio.load(audio_path)

        # Resample to 16 kHz if the clip was recorded at a different rate
        if orig_sr != 16000:
            audio = torchaudio.functional.resample(audio, orig_sr, 16000)

        # Convert stereo/multi-channel to mono by averaging across channels
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0)

        # Convert tensor to numpy array for the Seamless processor
        audio = audio.squeeze().numpy()

        # Prepare model inputs using the Seamless processor
        audio_inputs = seamless_large_processor(
            audio=audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        # Move all input tensors to the target device
        audio_inputs = {k: v.to(device) for k, v in audio_inputs.items()}

        with torch.no_grad():
            # Generate Urdu transcription tokens; disable speech output
            output_tokens = seamless_large_model.generate(
                **audio_inputs,
                tgt_lang="urd",
                generate_speech=False
            )

        # Decode token IDs back to a readable Urdu string
        transcription = seamless_large_processor.decode(
            output_tokens[0].tolist()[0],
            skip_special_tokens=True
        )

        seamless_large_results.append({
            "path": row["path"],
            "ground_truth": ground_truth,
            "prediction": transcription
        })

    except Exception as e:
        print(f"Error on {row['path']}: {e}")

        # Record error placeholder so the CSV row count stays aligned with test_df
        seamless_large_results.append({
            "path": row["path"],
            "ground_truth": ground_truth,
            "prediction": "[ERROR]"
        })

# Persist results to disk before releasing the model
seamless_large_df = pd.DataFrame(seamless_large_results)
seamless_large_df.to_csv("seamless_large_results.csv", index=False)

print(f"\n✓ Results saved: seamless_large_results.csv ({len(seamless_large_results)} transcriptions)")

# Delete model and free GPU memory before loading the next model
del seamless_large_model
del seamless_large_processor
torch.cuda.empty_cache()

print("✓ Model deleted and memory cleared")

## 13. Model 2 — Whisper Large v3 (Resumable)

**Model:** `openai/whisper-large-v3`  
**Task:** Urdu transcription with forced language/task decoder prompt

This cell is designed as a **resumable checkpoint runner** to handle kernel interruptions or CUDA errors gracefully:
1. Resets the CUDA context to clear any lingering memory issues
2. Loads any already-completed results from `whisper_large_results.csv`
3. Filters `test_df` to only the unprocessed clips
4. Loads the model with a CPU fallback if the GPU is unavailable
5. Runs inference and **checkpoints every 100 samples** by appending to the CSV
6. Verifies final row count matches `test_df` on completion

In [ ]:
print("\n" + "="*80)
print("RESETTING CUDA AND CONTINUING WHISPER-LARGE")
print("="*80)

import gc

# --- Step 1: Aggressive CUDA cleanup to free any stale memory allocations ---
print("Clearing CUDA context...")
torch.cuda.empty_cache()
gc.collect()

try:
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    print("✓ CUDA synchronized")
except:
    print("⚠ Could not fully reset CUDA - continuing anyway")

# --- Step 2: Load already-completed results to support mid-run resumption ---
existing_results = pd.read_csv('whisper_large_results.csv')
completed_paths = set(existing_results['path'].values)

print(f"✓ Found {len(existing_results)} completed examples")

# --- Step 3: Filter test_df to only clips that have not yet been processed ---
remaining_df = test_df[~test_df['path'].isin(completed_paths)].copy()
print(f"✓ Remaining examples to process: {len(remaining_df)}")

# --- Step 4: Load Whisper Large v3 with CPU fallback on CUDA failure ---
print("\nReloading Whisper-Large...")

try:
    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
    model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-large-v3",
        torch_dtype=torch.float32
    )

    # Attempt to place the model on GPU; fall back to CPU on failure
    try:
        model = model.to(device)
        print(f"✓ Model loaded on {device}")
    except RuntimeError as e:
        print(f"⚠ Cannot use GPU (CUDA corrupted): {e}")
        print("  Falling back to CPU (will be slower)...")
        device_fallback = "cpu"
        model = model.to(device_fallback)
        print(f"✓ Model loaded on CPU")
        device = device_fallback  # Update device reference for the inference loop

    model.eval()

except Exception as e:
    print(f"✗ FATAL: Cannot load model: {e}")
    print("\n🚨 YOU MUST RESTART THE KERNEL")
    print("   Then re-run from the continuation cell")
    raise

print("✓ Model ready\n")

# --- Step 5: Run inference on remaining (unprocessed) clips ---
print("Running inference on remaining examples...")
results = []

for idx, row in tqdm(remaining_df.iterrows(), total=len(remaining_df)):
    audio_path = f"{audio_folder}{row['path']}"
    ground_truth = row['sentence']

    try:
        # Load and resample audio to 16 kHz using librosa
        audio_array, sr = librosa.load(audio_path, sr=16000)
        inputs = processor(audio_array, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Force Urdu language and transcription task via Whisper's decoder prompt
        forced_decoder_ids = processor.get_decoder_prompt_ids(language="urdu", task="transcribe")

        with torch.no_grad():
            generated_ids = model.generate(
                inputs["input_features"],
                forced_decoder_ids=forced_decoder_ids,
                max_length=1024,
                num_beams=5,
                length_penalty=1.0
            )

        transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': transcription
        })

    except RuntimeError as e:
        # Handle CUDA runtime errors (e.g., OOM) and clear cache to recover
        print(f"\nRuntime error on {row['path']}: {e}")
        results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': "[ERROR]"
        })
        if device == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"\nError on {row['path']}: {e}")
        results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': "[ERROR]"
        })

    # Checkpoint: flush in-memory results to disk every 100 samples
    # This prevents total data loss if the kernel is interrupted
    if len(results) % 100 == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv('whisper_large_results.csv', mode='a', header=False, index=False)
        print(f"\n  ✓ Checkpoint: {len(results)} new examples")
        results = []  # Clear buffer after successful flush

# --- Step 6: Flush any remaining results not yet checkpointed ---
if len(results) > 0:
    final_df = pd.DataFrame(results)
    final_df.to_csv('whisper_large_results.csv', mode='a', header=False, index=False)
    print(f"\n✓ Final batch: {len(results)} examples")

# --- Step 7: Verify the output file is complete ---
final_results = pd.read_csv('whisper_large_results.csv')
print(f"\n{'='*80}")
print(f"COMPLETION CHECK")
print(f"{'='*80}")
print(f"  Total in CSV: {len(final_results)}")
print(f"  Expected: {len(test_df)}")
print(f"  Status: {'✓ COMPLETE' if len(final_results) == len(test_df) else '⚠ INCOMPLETE'}")

# Free GPU memory before the next model
del model
del processor
if device == "cuda":
    torch.cuda.empty_cache()
print("\n✓ Model deleted")

## 14. Model 3 — Whisper Medium

**Model:** `openai/whisper-medium`  
**Task:** Urdu transcription with forced decoder prompt

A lighter alternative to Whisper Large — useful as a speed/accuracy tradeoff data point. Uses the same `forced_decoder_ids` approach for Urdu language forcing.

In [ ]:
print("\n" + "="*80)
print("MODEL 4: Whisper-Medium")
print("="*80)

# Load the Whisper Medium processor and model
print("Loading Whisper-Medium...")
whisper_medium_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
whisper_medium_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
whisper_medium_model = whisper_medium_model.to(device)
whisper_medium_model.eval()
print("✓ Model loaded\n")

# Run inference across the full test set
print("Running inference...")
whisper_medium_results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = f"{audio_folder}{row['path']}"
    ground_truth = row['sentence']

    try:
        # Load and resample audio to 16 kHz using librosa
        audio_array, sr = librosa.load(audio_path, sr=16000)

        # Prepare Whisper input features from the raw audio array
        inputs = whisper_medium_processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        )
        inputs = inputs.to(device)

        # Force Urdu language and transcription task via Whisper's decoder prompt
        forced_decoder_ids = whisper_medium_processor.get_decoder_prompt_ids(language="urdu", task="transcribe")

        with torch.no_grad():
            generated_ids = whisper_medium_model.generate(
                inputs["input_features"],
                forced_decoder_ids=forced_decoder_ids
            )

        transcription = whisper_medium_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        whisper_medium_results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': transcription
        })

    except Exception as e:
        print(f"Error on {row['path']}: {e}")
        whisper_medium_results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': "[ERROR]"
        })

# Persist results before releasing the model
whisper_medium_df = pd.DataFrame(whisper_medium_results)
whisper_medium_df.to_csv('whisper_medium_results.csv', index=False)
print(f"\n✓ Results saved: whisper_medium_results.csv ({len(whisper_medium_results)} transcriptions)")

# Free GPU memory before loading the next model
del whisper_medium_model
del whisper_medium_processor
torch.cuda.empty_cache()
print("✓ Model deleted and memory cleared")

## 15. Model 4 — Wav2Vec2 Urdu (XLS-R 300M)

**Model:** `kingabzpro/wav2vec2-large-xls-r-300m-Urdu`  
**Task:** CTC-based Urdu speech recognition

Unlike the encoder-decoder Whisper and Seamless models, Wav2Vec2 uses a **CTC (Connectionist Temporal Classification)** head:
- Outputs per-frame log-probability logits over the vocabulary
- Decodes via `argmax` (greedy) rather than beam search
- No language forcing required — the model is already fine-tuned on Urdu

In [ ]:
print("\n" + "="*80)
print("MODEL 5: Wav2Vec2-Urdu")
print("="*80)

# Load the Urdu-fine-tuned Wav2Vec2 processor and CTC model
print("Loading Wav2Vec2-Urdu...")
wav2vec_processor = Wav2Vec2Processor.from_pretrained("kingabzpro/wav2vec2-large-xls-r-300m-Urdu")
wav2vec_model = Wav2Vec2ForCTC.from_pretrained("kingabzpro/wav2vec2-large-xls-r-300m-Urdu")
wav2vec_model = wav2vec_model.to(device)
wav2vec_model.eval()
print("✓ Model loaded\n")

# Run inference across the full test set
print("Running inference...")
wav2vec_results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = f"{audio_folder}{row['path']}"
    ground_truth = row['sentence']

    try:
        # Load and resample audio to 16 kHz using librosa
        audio_array, sr = librosa.load(audio_path, sr=16000)

        # Prepare Wav2Vec2 inputs; padding=True handles variable-length sequences
        inputs = wav2vec_processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )
        inputs = inputs.to(device)

        with torch.no_grad():
            # Forward pass: obtain per-frame log-probability logits from CTC head
            logits = wav2vec_model(inputs.input_values).logits

        # Greedy CTC decode: take the highest-probability token at each time step
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = wav2vec_processor.batch_decode(predicted_ids)[0]

        wav2vec_results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': transcription
        })

    except Exception as e:
        print(f"Error on {row['path']}: {e}")
        wav2vec_results.append({
            'path': row['path'],
            'ground_truth': ground_truth,
            'prediction': "[ERROR]"
        })

# Persist results before releasing the model
wav2vec_df = pd.DataFrame(wav2vec_results)
wav2vec_df.to_csv('wav2vec_urdu_results.csv', index=False)
print(f"\n✓ Results saved: wav2vec_urdu_results.csv ({len(wav2vec_results)} transcriptions)")

# Free GPU memory before the aggregation step
del wav2vec_model
del wav2vec_processor
torch.cuda.empty_cache()
print("✓ Model deleted and memory cleared")

## 16. Aggregate All Model Results

Load the four per-model CSVs and merge them into a single wide-format DataFrame for downstream evaluation (WER, CER, etc.).

| Column | Description |
|---|---|
| `path` | Audio clip filename |
| `ground_truth` | Reference Urdu sentence |
| `seamless_large` | Seamless M4T v2 Large prediction |
| `whisper_large` | Whisper Large v3 prediction |
| `whisper_medium` | Whisper Medium prediction |
| `wav2vec_urdu` | Wav2Vec2 Urdu (XLS-R 300M) prediction |

In [ ]:
print("\n" + "="*80)
print("COMBINING ALL MODEL RESULTS")
print("="*80)

# Load each model's per-clip prediction CSV from disk
seamless_large_df = pd.read_csv('seamless_large_results.csv')
whisper_large_df  = pd.read_csv('whisper_large_results.csv')
whisper_medium_df = pd.read_csv('whisper_medium_results.csv')
wav2vec_df        = pd.read_csv('wav2vec_urdu_results.csv')

# Build a wide-format DataFrame aligned on the test_df path order.
# All CSVs must share the same row order as test_df for this positional join to be valid.
combined_results = pd.DataFrame({
    'path'          : test_df['path'].values,
    'ground_truth'  : test_df['sentence'].values,
    'seamless_large': seamless_large_df['prediction'].values,
    'whisper_large' : whisper_large_df['prediction'].values,
    'whisper_medium': whisper_medium_df['prediction'].values,
    'wav2vec_urdu'  : wav2vec_df['prediction'].values
})

# Save the combined results for WER/CER evaluation
combined_results.to_csv('all_models_combined.csv', index=False)
print(f"✓ All results combined: all_models_combined.csv")
print(f"  Total samples: {len(combined_results)}")
print("\n🎉 Pipeline complete!")